In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import math

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


In [3]:
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

100%|██████████| 4.54k/4.54k [00:00<00:00, 3.66MB/s]


In [4]:
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [5]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
class patchembedding(nn.Module):
       def __init__(self,img_size=28,patch_size=7,in_channels=1,embed_dim=64):
              super().__init__()
              self.patch_size=patch_size
              self.num_patches=(img_size//patch_size)**2
              self.projection=nn.conv2d(in_channels,embed_dim,kernel_size=patch_size,stride=patch_size)
       def forward(self,x):
              x=self.projection(x)
              x=x.flatten(2)
              x=x.transpose(1,2)

multi head self attention 

In [9]:
class multiheadselfattention(nn.Module):
       def __init__(self,embed_dim=64,num_heads=4):
              super().__init__()
              assert embed_dim % num_heads == 0
              self.embed_dim=embed_dim
              self.num_heads=num_heads
              self.head_dim=embed_dim // num_heads
              self.qkv=nn.Linear(embed_dim,embed_dim*3)
              self.out=nn.Linear(embed_dim,embed_dim)
              def forward(self,x):
                     x=self.projection(x)
                     x=x.flatten(2)
                     x=x.transpose(1,2)

In [10]:
class MultiHeadSelfAttention(nn.Module):
       def __init__(self,embed_dim=64,num_heads=4):
              super().__init__()
              self.embed_dim=embed_dim
              self.num_heads=num_heads
              self.head_dim=embed_dim // num_heads
              self.qkv=nn.Linear(embed_dim,embed_dim*3)
              self.out=nn.Linear(embed_dim,embed_dim)

              def forward(self,x):
                     B,S,E,=x.shape
                     qkv=self.qkv(x)
                     qkv=qkv.reshape(B,S,3,self.num_heads,self.head_dim)
                     qkv=qkv.permute(2,0,3,1,4)
                     q,k,v=qkv[0],qkv[1],qkv[2]
                     scores=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.head_dim)
                     attn=F.softmax(scores,dim=-1)
                     out=torch.matmul(attn,v)
                     out=out.transpose(1,2).reshape(B,S,self.embed_dim)
                     out=self.out(out)
                     return out,attn


transformer encoder block

In [11]:
class TransformerEncodersBlock(nn.Module):
       def __init__(self,embed_dim=64,num_heads=4,mlp_dim=128,dropout=0.1):
              super().__init__()
              self.norm1=nn.LayerNorm(embed_dim)
              self.attn=MultiHeadSelfAttention(embed_dim,num_heads)
              self.norm2=nn.LayerNorm()

              self.mlp=nn.sequential(
                     nn.Linear(embed_dim,mlp_dim),
                     nn.GELU(),
                     nn.Dropout(dropout),
                     nn.Linear(mlp_dim,embed_dim),
                     nn.Dropout(dropout)
              )
              def forward(self,x):
                     x=x+self.attn(self.norm1(x))
                     x=x+self.mlp(self.norm2(x))
                     return x


full transformer block 

In [ ]:
class MNISTTransformer(nn.Module):
       def __init__(self,img_size=28,patch_size=7,in_channels=1,embed_dim=64,num_heads=4,mlp_dim=128,num_layers=6,num_classes=10,dropout=0.1):
              super().__init__()
              self.patch_embedding=patch_embedding(img_size,patch_size,in_channels,embed_dim)
              self.num_patches=self.patch_embedding.num_patches
              self.cls_token=nn.Parameter(torch.zeros(1,1,embed_dim))
              self.pos_embedding=nn.Parameter(torch.zeros(1,self.num_patches+1,embed_dim))
              self.dropout=nn.Dropout(dropout)
              self.encoder_blocks=nn.Sequential(*[TransformerEncodersBlock(embed_dim=embed_dim,num_heads=num_heads,mlp_dim=mlp_dim,dropout=dropout) for _ in range(num_layers)])
              self.norm=nn.LayerNorm(embed_dim)
              self.classifier=nn.Linear(embed_dim,num_classes)
       def forward(self,x):
              B=x.shape[0]
              x=self.patch_embedding(x)
              cls_tokens=self.cls_token.expand(B,-1,-1)
              x=torch.cat((cls_tokens,x),dim=1)
              x=x+self.pos_embedding
              x=self.dropout(x)
              x=self.encoder_blocks(x)
              x=self.norm(x)
              cls_output=x[:,0]
              logits=self.classifier(cls_output)
              return logits

